In [1]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from sqlalchemy import create_engine

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


In [2]:
## 1. Configuration
DB_USER = "root"
DB_PASSWORD = "root"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "capstone1"

OUTPUT_DIR = "ml3_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "ml3_best_model.pkl"
)

REPORT_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "ml3_evaluation_report.csv"
)

IMPORTANCE_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "ml3_feature_importance.csv"
)

In [4]:
## 2. Database Connection
connection_string = (
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(connection_string)

print("Database connected successfully")

Database connected successfully


In [5]:
## 3. Load Feature Data
feature_query = """
SELECT
    grid_id,
    feature_timestamp,
    avg_activity,
    median_activity,
    median_avg_ratio,
    activity_growth,
    active_hours,
    peak_ratio,
    variability,
    internet_share,
    hour_of_day,
    day_of_week
FROM network_feature_new_final
"""

features = pd.read_sql(feature_query, engine)

print(features.shape)
features.head()

(1209994, 12)


,grid_id,feature_timestamp,avg_activity,median_activity,median_avg_ratio,activity_growth,active_hours,peak_ratio,variability,internet_share,hour_of_day,day_of_week
0,1003,2013-11-02 23:00:00,24.776337,27.9924,1.129804,-0.139891,24,1.361061,7.473458,0.868073,4,1
1,1003,2013-11-03 00:00:00,24.880987,27.9924,1.125052,-0.132221,24,1.355336,7.417921,0.868109,5,1
2,1003,2013-11-03 01:00:00,24.928050,27.9924,1.122928,-0.121925,24,1.352777,7.371036,0.870391,6,1
3,1003,2013-11-03 02:00:00,25.069046,27.9924,1.116612,-0.107717,24,1.345169,7.187559,0.871087,7,1
4,1003,2013-11-03 03:00:00,25.014187,27.9924,1.119061,-0.105242,24,1.348119,7.269992,0.870833,8,1


In [6]:
## 4. Load Activity Data
activity_query = """
SELECT
    g.grid_id,
    t.timestamp,
    f.total_activity
FROM fact_network_activity f
INNER JOIN dim_time t
    ON f.time_key = t.time_key
INNER JOIN dim_grid g
    ON f.grid_key = g.grid_key
WHERE f.total_activity IS NOT NULL
ORDER BY g.grid_id,t.timestamp
"""

fact = pd.read_sql(activity_query, engine)

fact["timestamp"] = pd.to_datetime(
    fact["timestamp"]
)

fact.head()

,grid_id,timestamp,total_activity
0,1,2013-11-01 00:00:00,62.0092
1,1,2013-11-01 01:00:00,46.3654
2,1,2013-11-01 02:00:00,42.0870
3,1,2013-11-01 03:00:00,35.0978
4,1,2013-11-01 04:00:00,32.2741


In [7]:
engine.dispose()
print("Database connection closed")

Database connection closed


In [8]:
## 5. Create Next-Hour Target
features["feature_timestamp"] = pd.to_datetime(
    features["feature_timestamp"]
)

features["grid_id"] = features["grid_id"].astype(str)
fact["grid_id"] = fact["grid_id"].astype(str)

features["target_timestamp"] = (
    features["feature_timestamp"]
    + pd.Timedelta(hours=1)
)

fact = fact.rename(
    columns={
        "timestamp":"target_timestamp",
        "total_activity":"next_hour_activity"
    }
)

ml_data = features.merge(
    fact,
    on=["grid_id","target_timestamp"],
    how="left"
)

ml_data = ml_data.dropna(
    subset=["next_hour_activity"]
)

In [9]:
## 6. Train/Test Split
ml_data = ml_data.sort_values(
    "feature_timestamp"
).reset_index(drop=True)

unique_timestamps = (
    ml_data["feature_timestamp"]
    .drop_duplicates()
    .sort_values()
)

split_index = int(
    len(unique_timestamps) * 0.8
)

train_end = unique_timestamps.iloc[
    split_index - 1
]

test_start = unique_timestamps.iloc[
    split_index
]

train_data = ml_data[
    ml_data["feature_timestamp"] <= train_end
]

test_data = ml_data[
    ml_data["feature_timestamp"] >= test_start
]

In [10]:
## 7. Create HIGH_ACTIVITY Target
SURGE_FACTOR = 1.20

train_data["HIGH_ACTIVITY"] = (
    train_data["next_hour_activity"]
    >= train_data["median_activity"] * SURGE_FACTOR
).astype(int)

test_data["HIGH_ACTIVITY"] = (
    test_data["next_hour_activity"]
    >= test_data["median_activity"] * SURGE_FACTOR
).astype(int)

In [11]:
print("\nSTEP 15 — CLASS BALANCE")
print("-" * 70)

train_counts = train_data[
    "HIGH_ACTIVITY"
].value_counts()

test_counts = test_data[
    "HIGH_ACTIVITY"
].value_counts()

train_base_rate = train_data[
    "HIGH_ACTIVITY"
].mean()

test_base_rate = test_data[
    "HIGH_ACTIVITY"
].mean()

print("\nTRAIN")
print(
    train_data["HIGH_ACTIVITY"]
    .value_counts()
    .sort_index()
)

print(
    f"Training positive base rate: "
    f"{train_base_rate:.4f} "
    f"({train_base_rate * 100:.2f}%)"
)

print("\nTEST")
print(
    test_data["HIGH_ACTIVITY"]
    .value_counts()
    .sort_index()
)

print(
    f"Test positive base rate: "
    f"{test_base_rate:.4f} "
    f"({test_base_rate * 100:.2f}%)"
)




STEP 15 — CLASS BALANCE
----------------------------------------------------------------------

TRAIN
HIGH_ACTIVITY
0    656721
1    303270
Name: count, dtype: int64
Training positive base rate: 0.3159 (31.59%)

TEST
HIGH_ACTIVITY
0    172401
1     67597
Name: count, dtype: int64
Test positive base rate: 0.2817 (28.17%)


In [12]:
## 8. Feature Selection and Data Cleaning
FEATURE_COLUMNS = [
    "avg_activity",
    "median_activity",
    "median_avg_ratio",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share",
    "hour_of_day",
    "day_of_week"
]

TARGET_COLUMN = "HIGH_ACTIVITY"

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN]

X_test = test_data[FEATURE_COLUMNS]
y_test = test_data[TARGET_COLUMN]

X_train = X_train.replace(
    [np.inf,-np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf,-np.inf],
    np.nan
)

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

In [13]:
## 9. Train Logistic Regression
logistic_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train,y_train)

logistic_predictions = logistic_model.predict(X_test)

In [14]:
## 10. Train Decision Tree
# from sklearn.model_selection import ParameterGrid

# params = {
#     "max_depth": [5, 8, 10, 12, 15],
#     "min_samples_leaf": [5, 10, 20, 50],
#     "min_samples_split": [10, 20, 50]
# }
# best_f1 = -1

# for p in ParameterGrid(params):
best_d_f1 = -1
best_d_threshold = None
best_d_predictions = None
model = DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42,
        max_depth=15,
        min_samples_leaf= 50,
        min_samples_split= 10



    )

model.fit(X_train, y_train)

tree_probs = (
    model
    .predict_proba(X_test)[:,1]
)
for t in np.arange(0.05,0.95,0.05):

    pred = (tree_probs >= t).astype(int)

    f1 = f1_score(
        y_test,
        pred
    )

    precision = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    print(
        f"Threshold={t:.2f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={f1:.4f}"
    )

    if f1 > best_d_f1:
        best_d_f1 = f1
        best_d_threshold = t
        best_d_predictions = pred

print(
    f"\nBest XGB Threshold: "
    f"{best_d_threshold:.2f}"
)

print(
    f"Best XGB F1: "
    f"{best_d_f1:.4f}"
)

#     f1 = f1_score(y_test, pred)

#     if f1 > best_f1:
#         best_f1 = f1
#         best_params = p

# print(best_params)
# print(best_f1)
#{'max_depth': 15, 'min_samples_leaf': 50, 'min_samples_split': 10}
# decision_tree_model.fit(
#     X_train,
#     y_train
# )

# tree_predictions = decision_tree_model.predict(X_test)

Threshold=0.05 | Precision=0.4576 | Recall=0.9758 | F1=0.6230
Threshold=0.10 | Precision=0.4792 | Recall=0.9643 | F1=0.6402
Threshold=0.15 | Precision=0.4932 | Recall=0.9489 | F1=0.6491
Threshold=0.20 | Precision=0.5111 | Recall=0.9334 | F1=0.6605
Threshold=0.25 | Precision=0.5251 | Recall=0.9188 | F1=0.6683
Threshold=0.30 | Precision=0.5456 | Recall=0.9020 | F1=0.6799
Threshold=0.35 | Precision=0.5604 | Recall=0.8791 | F1=0.6845
Threshold=0.40 | Precision=0.5816 | Recall=0.8591 | F1=0.6936
Threshold=0.45 | Precision=0.5992 | Recall=0.8399 | F1=0.6994
Threshold=0.50 | Precision=0.6212 | Recall=0.8086 | F1=0.7026
Threshold=0.55 | Precision=0.6529 | Recall=0.7340 | F1=0.6911
Threshold=0.60 | Precision=0.6733 | Recall=0.7112 | F1=0.6918
Threshold=0.65 | Precision=0.6839 | Recall=0.6905 | F1=0.6872
Threshold=0.70 | Precision=0.7171 | Recall=0.6373 | F1=0.6748
Threshold=0.75 | Precision=0.7358 | Recall=0.5959 | F1=0.6585
Threshold=0.80 | Precision=0.7722 | Recall=0.5322 | F1=0.6301
Threshol

In [15]:
# ================================================================
# MODEL 4 — XGBOOST
# ================================================================

print("\nSTEP 21 — TRAINING XGBOOST")
print("-" * 70)

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train
)
# xgb_probs=xgb_model.predict(X_test)

# Predict probabilities
xgb_probs = xgb_model.predict_proba(
    X_test
)[:, 1]

# Use previously identified best threshold
best_xgb_threshold = 0.35   # replace with your value

xgb_predictions = (
    xgb_probs >= best_xgb_threshold
).astype(int)

print(
    f"Using XGBoost threshold: "
    f"{best_xgb_threshold:.2f}"
)

print("XGBoost training complete.")

# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score
# )
# import numpy as np

# Predict probabilities
# xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# print("\nXGBoost Threshold Analysis")
# print("-" * 80)

# best_threshold = None
# best_f1 = 0

# for threshold in np.arange(0.10, 0.95, 0.05):

#     preds = (xgb_probs >= threshold).astype(int)

#     accuracy = accuracy_score(y_test, preds)
#     precision = precision_score(y_test, preds, zero_division=0)
#     recall = recall_score(y_test, preds, zero_division=0)
#     f1 = f1_score(y_test, preds, zero_division=0)

#     print(
#         f"Threshold={threshold:.2f} | "
#         f"Accuracy={accuracy:.4f} | "
#         f"Precision={precision:.4f} | "
#         f"Recall={recall:.4f} | "
#         f"F1={f1:.4f}"
#     )

#     if f1 > best_f1:
#         best_f1 = f1
#         best_threshold = threshold

# print("\nBest Threshold")
# print(f"Threshold: {best_threshold:.2f}")
# print(f"Best F1 Score: {best_f1:.4f}")

# # Final predictions using best threshold
# xgb_predictions = (
#     xgb_probs >= best_threshold
# ).astype(int)


STEP 21 — TRAINING XGBOOST
----------------------------------------------------------------------


Using XGBoost threshold: 0.35
XGBoost training complete.


In [16]:
## 14. Evaluate Models
def evaluate_model(
    model_name,
    y_true,
    predictions,
    base_rate
):

    accuracy = accuracy_score(
        y_true,
        predictions
    )

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        predictions
    )

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"Base rate: {base_rate:.4f}"
    )

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            predictions,
            zero_division=0
        )
    )

    return {
        "model": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "base_rate": base_rate
    }


In [17]:

logistic_results = evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_predictions,
    test_base_rate
)


Logistic Regression
Accuracy : 0.7302
Precision: 0.5135
Recall   : 0.8027
Base rate: 0.2817

Confusion Matrix:
[[120991  51410]
 [ 13339  54258]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.70      0.79    172401
           1       0.51      0.80      0.63     67597

    accuracy                           0.73    239998
   macro avg       0.71      0.75      0.71    239998
weighted avg       0.79      0.73      0.74    239998



In [18]:
tree_results = evaluate_model(
    "Decision Tree",
    y_test,
    pred,
    test_base_rate
)


Decision Tree
Accuracy : 0.8026
Precision: 0.8354
Recall   : 0.3724
Base rate: 0.2817

Confusion Matrix:
[[167440   4961]
 [ 42425  25172]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.97      0.88    172401
           1       0.84      0.37      0.52     67597

    accuracy                           0.80    239998
   macro avg       0.82      0.67      0.70    239998
weighted avg       0.81      0.80      0.77    239998



In [19]:
xgb_results = evaluate_model(

"XGBoost",

y_test,

xgb_predictions,

test_base_rate

)


XGBoost
Accuracy : 0.8418
Precision: 0.6940
Recall   : 0.7840
Base rate: 0.2817

Confusion Matrix:
[[149031  23370]
 [ 14600  52997]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.86      0.89    172401
           1       0.69      0.78      0.74     67597

    accuracy                           0.84    239998
   macro avg       0.80      0.82      0.81    239998
weighted avg       0.85      0.84      0.84    239998



In [20]:
from sklearn.metrics import f1_score

results = pd.DataFrame(
    [
        logistic_results,
        tree_results,
        xgb_results
    ]
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions,
    zero_division=0
)

tree_predictions = (
    tree_probs >= 0.50
).astype(int)

tree_f1 = f1_score(
    y_test,
    tree_predictions,
    zero_division=0
)

xgb_f1 = f1_score(
    y_test,
    xgb_predictions,
    zero_division=0
)

results["f1"] = [
    logistic_f1,
    tree_f1,
    xgb_f1
]

print("\nModel comparison including F1:")
print(results.to_string(index=False))


Model comparison including F1:
              model  accuracy  precision   recall  base_rate       f1
Logistic Regression  0.730210   0.513476 0.802669   0.281657 0.626301
      Decision Tree  0.802557   0.835363 0.372383   0.281657 0.702599
            XGBoost  0.841790   0.693978 0.784014   0.281657 0.736254


In [21]:
## 16. Feature Importance
xgb_importance = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "importance":
            xgb_model.feature_importances_
    }
).sort_values(
    "importance",
    ascending=False
)

print("\nXGBOOST FEATURE IMPORTANCE")
print(xgb_importance)


XGBOOST FEATURE IMPORTANCE
            feature  importance
8       hour_of_day    0.471010
2  median_avg_ratio    0.155285
9       day_of_week    0.103999
5        peak_ratio    0.081501
3   activity_growth    0.048405
7    internet_share    0.042070
6       variability    0.036949
1   median_activity    0.031067
0      avg_activity    0.029713
4      active_hours    0.000000


In [22]:
## 17. Save Outputs
xgb_importance.to_csv(
    IMPORTANCE_OUTPUT,
    index=False
)

print(
    f"Saved: {IMPORTANCE_OUTPUT}"
)


Saved: ml3_output\ml3_feature_importance.csv


In [23]:

results["f1"] = [
    logistic_f1,
    tree_f1,
    xgb_f1
]
results.to_csv(
    REPORT_OUTPUT,
    index=False
)

print(
    f"Saved: {REPORT_OUTPUT}"
)

Saved: ml3_output\ml3_evaluation_report.csv


In [24]:
model_package = {
    "model": xgb_model,
    "model_name": "XGBoost",
    "features": FEATURE_COLUMNS,
    "target": TARGET_COLUMN,
    "threshold": best_xgb_threshold,
    # "grid_p85_thresholds": grid_p85.to_dict(),
    # "p85_threshold": p85_threshold,
    "train_start": train_data[
        "feature_timestamp"
    ].min(),
    "train_end": train_data[
        "feature_timestamp"
    ].max(),
    "test_start": test_data[
        "feature_timestamp"
    ].min(),
    "test_end": test_data[
        "feature_timestamp"
    ].max()
}

with open(
    MODEL_OUTPUT,
    "wb"
) as file:

    pickle.dump(
        model_package,
        file
    )

print(
    f"Saved: {MODEL_OUTPUT}"
)

Saved: ml3_output\ml3_best_model.pkl
